# Notebook 11 — Evaluación sobre el conjunto de test (2025)

Este notebook evalúa los modelos más representativos de cada familia
sobre el conjunto de test (2025), que ha permanecido completamente
intacto durante todo el desarrollo del TFM.

**Modelos evaluados:**
- Naive_seasonal — baseline de referencia
- SARIMA(1,1,1)×(1,0,1)[24] — mejor modelo estadístico clásico
- AutoReg B0 — autoreg puro (lags=336)
- AutoReg B6 — mejor modelo global (lags=336 + sin/cos hora + día semana)
- LSTM(h=128,l=2,lb=7d) — mejor modelo de aprendizaje profundo

**Nota metodológica:** los modelos estadísticos se reentrenan en cada
fit del walk-forward incorporando progresivamente los datos de 2025.
El LSTM se evalúa con los pesos fijos del entrenamiento final sobre
train+val (2022--2024), condición realista de despliegue en producción.

**Protocolo:** walk-forward con 45 fits, paso 8 días, primer cutoff
al inicio de 2025, IC 95% por bootstrap no paramétrico (1000 iteraciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.statespace.sarimax import SARIMAX
import tensorflow as tf
import warnings
warnings.filterwarnings("ignore")
BASE_DIR    = Path("/home/ubuntu/TFM")
RESULTS_DIR = BASE_DIR / "results"
FIG_DIR     = BASE_DIR / "figures"
DATA_DIR    = BASE_DIR / "notebooks/data_processed"

SEASONAL_PERIOD = 24
HORIZONS        = {"48h": 48, "72h": 72}
LAGS_OPT        = 336
STEP            = 8 * SEASONAL_PERIOD
N_FITS          = 45
N_BOOTSTRAP     = 1000
SEED            = 42

np.random.seed(SEED)
print("Imports OK")

## 1. Carga de datos

In [ ]:
# Cargar serie completa
df_full = pd.read_parquet(
    DATA_DIR / "es_carbon_footprint_operational_global_1h_2022_2025.parquet"
)

# Separar conjuntos
train_val = df_full[df_full.index.year < 2025]["y"].values
test      = df_full[df_full.index.year == 2025]["y"].values
y_full    = np.concatenate([train_val, test])

# Índices temporales
dates_full    = df_full.index
dates_train_val = df_full[df_full.index.year < 2025].index
dates_test    = df_full[df_full.index.year == 2025].index

n_train_val = len(train_val)

print(f"Train+Val: {len(train_val)} obs | {dates_train_val[0].date()} → {dates_train_val[-1].date()}")
print(f"Test:      {len(test)} obs | {dates_test[0].date()} → {dates_test[-1].date()}")
print(f"Total:     {len(y_full)} obs")

## 2. Variables exógenas B6 (sin/cos hora + día de semana)

In [ ]:
def build_exog_b6(dates):
    """Variables cíclicas de tiempo para AutoReg B6."""
    return np.column_stack([
        np.sin(2 * np.pi * dates.hour / 24),
        np.cos(2 * np.pi * dates.hour / 24),
        np.sin(2 * np.pi * dates.dayofweek / 7),
        np.cos(2 * np.pi * dates.dayofweek / 7)
    ])

exog_full = build_exog_b6(dates_full)

print(f"Exógenas shape: {exog_full.shape}")
print("Variables: sin_hour, cos_hour, sin_dow, cos_dow")

## 3. Funciones auxiliares

In [ ]:
def compute_metrics(y_true, y_pred):
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    return mae, rmse

def bootstrap_ci(errors, n=N_BOOTSTRAP, seed=SEED):
    rng   = np.random.default_rng(seed)
    means = [rng.choice(errors, len(errors), replace=True).mean()
             for _ in range(n)]
    return float(np.mean(errors)), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

def summarize_results(df_results, model_name):
    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    for h in ["48h", "72h"]:
        errs = df_results[df_results["horizon"] == h]["MAE"].values
        mean, lo, hi = bootstrap_ci(errs)
        std = errs.std()
        print(f"  {h}:  MAE = {mean:.2f} ± {std:.2f}  IC95% [{lo:.2f}, {hi:.2f}]  (n={len(errs)})")

print("Funciones definidas OK")

## 4. Walk-forward sobre test set

El walk-forward avanza desde el final de train+val (diciembre 2024)
hasta el final de 2025, con paso de 8 días y 45 fits.

In [ ]:
print("Evaluando Naive_seasonal...")
naive_results = []

cutoff = n_train_val
for fit_i in range(N_FITS):
    if cutoff + max(HORIZONS.values()) > len(y_full):
        break
    for h_name, h in HORIZONS.items():
        # Naive seasonal: copia las últimas 24 horas
        pred   = np.tile(y_full[cutoff-24:cutoff], h//24 + 1)[:h]
        y_true = y_full[cutoff:cutoff+h]
        mae, rmse = compute_metrics(y_true, pred)
        naive_results.append({
            "fit": fit_i+1, "horizon": h_name,
            "MAE": mae, "RMSE": rmse
        })
    cutoff += STEP

df_naive = pd.DataFrame(naive_results)
summarize_results(df_naive, "Naive_seasonal (TEST)")

In [ ]:
print("Evaluando SARIMA(1,1,1)x(1,0,1,24)...")
sarima_results = []

cutoff = n_train_val
for fit_i in range(N_FITS):
    if cutoff + max(HORIZONS.values()) > len(y_full):
        break

    y_train = y_full[:cutoff]
    print(f"  Fit {fit_i+1}/{N_FITS} — cutoff {dates_full[cutoff].date()}")

    try:
        model = SARIMAX(
            y_train[-90*24:],  # últimos 90 días para velocidad
            order=(1,1,1),
            seasonal_order=(1,0,1,24),
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)

        for h_name, h in HORIZONS.items():
            pred   = model.forecast(steps=h)
            y_true = y_full[cutoff:cutoff+h]
            mae, rmse = compute_metrics(y_true, pred)
            sarima_results.append({
                "fit": fit_i+1, "horizon": h_name,
                "MAE": mae, "RMSE": rmse
            })
    except Exception as e:
        print(f"    Error en fit {fit_i+1}: {e}")

    cutoff += STEP

df_sarima = pd.DataFrame(sarima_results)
summarize_results(df_sarima, "SARIMA(1,1,1)x(1,0,1,24) (TEST)")

In [ ]:
print("Evaluando AutoReg B0 (lags=336)...")
b0_results = []

cutoff = n_train_val
for fit_i in range(N_FITS):
    if cutoff + max(HORIZONS.values()) > len(y_full):
        break

    y_train = y_full[:cutoff]

    for h_name, h in HORIZONS.items():
        model = AutoReg(y_train, lags=LAGS_OPT, trend='c').fit()
        pred  = model.predict(start=cutoff, end=cutoff+h-1)
        y_true = y_full[cutoff:cutoff+h]
        mae, rmse = compute_metrics(y_true, pred)
        b0_results.append({
            "fit": fit_i+1, "horizon": h_name,
            "MAE": mae, "RMSE": rmse
        })

    cutoff += STEP
    if fit_i % 10 == 0:
        print(f"  Fit {fit_i+1}/{N_FITS}")

df_b0 = pd.DataFrame(b0_results)
summarize_results(df_b0, "AutoReg B0 (TEST)")

In [ ]:
print("Evaluando AutoReg B6 (lags=336 + sin/cos hora + día semana)...")
b6_results = []

cutoff = n_train_val
for fit_i in range(N_FITS):
    if cutoff + max(HORIZONS.values()) > len(y_full):
        break

    y_train  = y_full[:cutoff]
    e_train  = exog_full[:cutoff]

    for h_name, h in HORIZONS.items():
        e_future = exog_full[cutoff:cutoff+h]
        model = AutoReg(y_train, lags=LAGS_OPT,
                       exog=e_train, trend='c').fit()
        pred  = model.predict(start=cutoff, end=cutoff+h-1,
                             exog_oos=e_future)
        y_true = y_full[cutoff:cutoff+h]
        mae, rmse = compute_metrics(y_true, pred)
        b6_results.append({
            "fit": fit_i+1, "horizon": h_name,
            "MAE": mae, "RMSE": rmse
        })

    cutoff += STEP
    if fit_i % 10 == 0:
        print(f"  Fit {fit_i+1}/{N_FITS}")

df_b6 = pd.DataFrame(b6_results)
summarize_results(df_b6, "AutoReg B6 (TEST)")

## 5. Evaluación LSTM en test

El LSTM se evalúa con los pesos fijos del entrenamiento final
sobre train+val. Se carga el modelo guardado y se aplica
directamente sobre las secuencias del test set.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

print("Evaluando LSTM (pesos fijos del entrenamiento final)...")

# Cargar modelos guardados
lstm_48h = tf.keras.models.load_model(BASE_DIR / "models/lstm_48h.keras")
lstm_72h = tf.keras.models.load_model(BASE_DIR / "models/lstm_72h.keras")

# Scaler ajustado sobre train (2022-2023)
train_only = df_full[df_full.index.year < 2024]["y"].values
scaler = MinMaxScaler()
scaler.fit(train_only.reshape(-1,1))

# Serie completa escalada
y_scaled = scaler.transform(y_full.reshape(-1,1)).flatten()

BEST_LOOKBACK = 7 * SEASONAL_PERIOD  # 168

def create_sequence(y_scaled, cutoff, lookback, h):
    X = y_scaled[cutoff-lookback:cutoff].reshape(1, lookback, 1)
    return X

lstm_results = []
cutoff = n_train_val

for fit_i in range(N_FITS):
    if cutoff + max(HORIZONS.values()) > len(y_full):
        break

    for h_name, h, model in [
        ("48h", 48, lstm_48h),
        ("72h", 72, lstm_72h)
    ]:
        X = create_sequence(y_scaled, cutoff, BEST_LOOKBACK, h)
        pred_scaled = model.predict(X, verbose=0).flatten()
        pred = scaler.inverse_transform(
            pred_scaled.reshape(-1,1)
        ).flatten()
        y_true = y_full[cutoff:cutoff+h]
        mae, rmse = compute_metrics(y_true, pred)
        lstm_results.append({
            "fit": fit_i+1, "horizon": h_name,
            "MAE": mae, "RMSE": rmse
        })

    cutoff += STEP
    if fit_i % 10 == 0:
        print(f"  Fit {fit_i+1}/{N_FITS}")

df_lstm = pd.DataFrame(lstm_results)
summarize_results(df_lstm, "LSTM(h=128,l=2,lb=7d) (TEST)")

## 6. Tabla comparativa test vs validación

In [ ]:
# Resultados de validación (ya conocidos)
val_results = {
    "Naive_seasonal": {"48h": (20.84, 16.83, 25.59), "72h": (23.01, 18.37, 28.50)},
    "SARIMA":         {"48h": (22.21, 17.93, 27.24), "72h": (26.80, 21.59, 32.94)},
    "AutoReg_B0":     {"48h": (17.39, 13.58, 22.32), "72h": (18.98, 14.77, 24.02)},
    "AutoReg_B6":     {"48h": (17.02, 13.33, 21.62), "72h": (18.81, 14.88, 23.51)},
    "LSTM":           {"48h": (20.25, 17.08, 24.42), "72h": (23.52, 19.91, 28.01)},
}

# Resultados de test
test_dfs = {
    "Naive_seasonal": df_naive,
    "SARIMA":         df_sarima,
    "AutoReg_B0":     df_b0,
    "AutoReg_B6":     df_b6,
    "LSTM":           df_lstm,
}

print("\n========== COMPARACIÓN VALIDACIÓN vs TEST ==========\n")
print(f"{'Modelo':<20} {'H':>4} {'MAE Val':>8} {'MAE Test':>9} {'Δ':>7}")
print("-" * 52)

test_summary = []
for model_name, df_res in test_dfs.items():
    for h in ["48h", "72h"]:
        errs = df_res[df_res["horizon"] == h]["MAE"].values
        mean_test, lo, hi = bootstrap_ci(errs)
        mean_val = val_results[model_name][h][0]
        delta = mean_test - mean_val
        sign = "+" if delta > 0 else ""
        print(f"{model_name:<20} {h:>4} {mean_val:>8.2f} "
              f"{mean_test:>9.2f} {sign}{delta:>6.2f}")
        test_summary.append({
            "model": model_name, "horizon": h,
            "MAE_val": mean_val,
            "MAE_test": round(mean_test, 2),
            "IC_lo": round(lo, 2),
            "IC_hi": round(hi, 2),
            "delta": round(delta, 2)
        })

df_summary = pd.DataFrame(test_summary)
df_summary.to_csv(RESULTS_DIR / "test_results.csv", index=False)
print("\nResultados guardados en test_results.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    "Comparación MAE — Validación (2024) vs Test (2025)",
    fontsize=13, fontweight="bold"
)

models_order = ["Naive_seasonal", "SARIMA", "AutoReg_B0",
                "AutoReg_B6", "LSTM"]
colors_val  = "#457b9d"
colors_test = "#e76f51"
x = np.arange(len(models_order))
width = 0.35

for ax, h in zip(axes, ["48h", "72h"]):
    sub = df_summary[df_summary["horizon"] == h]
    sub = sub.set_index("model").loc[models_order]

    bars_val  = ax.bar(x - width/2, sub["MAE_val"],
                       width, label="Validación 2024",
                       color=colors_val, alpha=0.85)
    bars_test = ax.bar(x + width/2, sub["MAE_test"],
                       width, label="Test 2025",
                       color=colors_test, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(models_order, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("MAE (gCO₂eq/kWh)")
    ax.set_title(f"Horizonte {h}")
    ax.legend()
    ax.grid(axis="y", linestyle=":", alpha=0.4)

plt.tight_layout()
plt.savefig(FIG_DIR / "test_vs_val.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: test_vs_val.png")

## 7. Conclusiones de la evaluación en test

*(Actualizar con los resultados reales una vez ejecutado)*

La evaluación sobre el conjunto de test (2025) permite confirmar
si los resultados obtenidos en validación (2024) se generalizan
a datos completamente no vistos. Se analiza especialmente si el
ranking de modelos se mantiene y si el AutoReg B6 confirma su
posición como mejor modelo.